In [ ]:
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env.local")

import os

os.environ["HF_HUB_CACHE"] = "./models/huggingface/_cache"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"


In [ ]:
import torch

from diffusers import AutoPipelineForText2Image
from pathlib import Path
from typing import List, TypedDict

In [ ]:
def get_local_text_to_image_pipeline(
        model_id: str,
        models_folder: str = "./models/huggingface",
        force_redownload: bool = False,
):
    """
    Download a zero-shot model into this workspace and return an inference pipeline.

    Model files are stored at:
        {workspace_root}/{models_folder}/{model_id}
    """
    local_dir = Path(models_folder) / model_id.replace("/", "--")
    local_dir.mkdir(parents=True, exist_ok=True)

    has_model_files = (local_dir / "config.json").exists()

    if force_redownload or not has_model_files:
        AutoPipelineForText2Image.from_pretrained(model_id).save_pretrained(local_dir)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    print(f"Loading model \"{model_id}\" from \"{local_dir}\" on {device}")

    pipeline = AutoPipelineForText2Image.from_pretrained(local_dir, local_files_only=True, torch_dtype=dtype)
    pipeline.to(device)

    return pipeline

In [ ]:
class Experiment(TypedDict):
    title: str
    prompt: str


experiments: List[Experiment] = [
    {
        'title': 'Text rendering poster',
        'prompt': 'A modern coffee shop poster on a sidewalk that clearly reads "Open Late" and "Fresh Pastries", warm lighting, realistic photography',
    },
    {
        'title': 'Product shot',
        'prompt': 'A premium wristwatch on a matte black pedestal, studio lighting, subtle reflections, luxury product photography',
    },
    {
        'title': 'Concept art',
        'prompt': 'A rain-soaked cyberpunk alley with neon signs, cinematic framing, highly detailed environment concept art',
    }
]

In [ ]:
def run_image_generation_experiment(model_id: str):
    pipeline = get_local_text_to_image_pipeline(model_id)

    for i in range(len(experiments)):
        print(f"{'=' * 10} [{model_id}] {'=' * 10}")
        print(experiments[i]["title"])

        generation_result = pipeline(experiments[i]["prompt"], num_inference_steps=5)
        for image in generation_result.images:
            display(image)

In [ ]:
run_image_generation_experiment("stabilityai/sdxl-turbo")